# Portfolio and CBU reserve plan to the end of 2027

## The idea
The reserve is not planned on its own. **It follows the portfolio:** a sum of money in category 3 needs a category-3
reserve, wherever it came from. So we plan the **portfolio by quality category** first, and the reserve comes out of it.

Everything is done separately for every **segment = currency (UZS / FCY) × product**. Products behave differently
(a mortgage and a microloan go bad at very different speeds), and so do currencies.

## The two formulas
For one segment, let $S_t = (S_t^1, \dots, S_t^5)$ be the money on balance in categories 1…5 on date $t$.

$$S_{t+1} \;=\; S_t \cdot M \;+\; N_{t+1}$$

* $M$ is the **migration matrix** (5 rows × 7 columns). Row $i$ says where one sum of category-$i$ money is one month
  later: in category 1, 2, 3, 4, 5, **repaid**, or **written off**. Every row adds up to 100%.
* $N_{t+1}$ is the **new money** that month: planned issuance (выдачи) × the share of it that is still on balance at
  the end of the month, split by category the way new money has been split in the past.

$$\text{Reserve}_t \;=\; \sum_{i=1}^{5} S_t^i \cdot r^i$$

* $r^i$ is the **CBU reserve rate** of category $i$ (`CBU_MIN_RATE`). The same rule is applied to the actual
  months too, so actual and plan are comparable. Written-off money is off balance and carries no reserve.

**One exception: problem corporate loans are planned loan by loan, not by the matrix.** A corporate department has a
handful of big loans; an average cannot split one loan into "13% written off, 20% cured". Each such loan follows its
own overdue days (section 6), and you can write your own decision for any of them in `reserve_plan_corporate.xlsx`.

Applying the first formula month after month, from the last actual date to the end of 2027, gives the portfolio plan.
The second formula turns every month of it into the reserve plan.

## Why it is built this way
* **Migration instead of "NPL share × portfolio".** An NPL ratio says nothing about *where the money moves*. The matrix
  keeps the full story: loans going bad (1→2→3), loans curing (3→1), repayment, and write-off. Each of these can be
  changed separately in a scenario.
* **Money, not loan counts.** One 10-billion corporate loan matters more than a hundred microloans, so every
  share is weighted by balance.
* **Only the portfolio is forecast; the reserve is a fixed rule on top of it.** The reserve rate per category is the
  CBU rule, not a statistic, so all the uncertainty sits in one place (the migration) and the reserve cannot add noise
  of its own. Section 4 compares the rule with the reserve actually booked.
* **A backtest before any plan.** The same machinery is placed some months back and asked to predict today. If it cannot
  predict months we already know, its 2027 is not worth presenting.

## How to use it
1. Change only the **SETTINGS** cell, then *Run All*.
2. The first run creates two files you may edit, then run again:
   * `reserve_plan_vidacha.xlsx`: issuance (выдачи) per segment and month. Default = recent average; a product that
     issued nothing in the last `STOPPED_MONTHS` months gets 0 (it has stopped lending).
   * `reserve_plan_corporate.xlsx`: the problem corporate loans, one row each, with the automatic path and empty
     columns for your decision (repaid, written off, moved to a category) and its date.
3. Read the **checks (section 7)** before the plan. Then open `reserve_plan.xlsx`: sheet «План» is the template
   (portfolio on top, reserve below); the other sheets show the matrices, rates, flows, checks and settings.

In [ ]:
# ═══════════════════════════════ SETTINGS ═══════════════════════════════
from pathlib import Path

DATA_DIR = Path("data")                       # every *.parquet in this folder is read
OUT_FILE = Path("reserve_plan.xlsx")          # the result
LENDING_FILE = Path("reserve_plan_vidacha.xlsx")      # issuance plan: created on the first run, edit and rerun
CORPORATE_FILE = Path("reserve_plan_corporate.xlsx")  # problem corporate loans: created on the first run, edit and rerun

# Column names in the data (right-hand side = the name in your file)
COLS = dict(
    loan="anketa", date="report_date", currency="valuta", department="department",
    passport="passport", category="kategoriya_npl_90",
    balance="brutto_summa",            # on balance, in sum (FX already converted)
    reserve="reserve_summa",           # booked CBU reserve
    written_off="spisat_95413_summa",  # written off, cumulative: its growth = this month's write-off
    days="maks_dni",                   # days overdue
    client="contragent",               # the borrower
    issued="vidan_summa",              # issued (выдача)
)

# Dates are report dates: 01.01.2026 = the portfolio on 01.01.2026 (end of December 2025)
FACT_FROM = "2026-01-01"     # first actual column of the table
PLAN_TO = "2028-01-01"       # last plan column: 01.01.2028 = end of 2027

UZS_CODES = {"0", "000", "860", "UZS", "СУМ", "SUM", "SO'M"}   # these are UZS, everything else is FCY
CURRENCY_ORDER = ["UZS", "FCY"]

ONE_PRODUCT = {"mk": "MCB", "kk": "Корпоратив"}   # a whole department = one product; other departments → passport
PASSPORT_GROUPS = {}         # {"passport in data": "group"}; empty = passport as it is
PRODUCT_ORDER = []           # row order of the table; empty = largest first

CATEGORIES = [1, 2, 3, 4, 5]
NPL_CATEGORIES = [3, 4, 5]
CBU_MIN_RATE = {1: 0.0, 2: 0.10, 3: 0.25, 4: 0.50, 5: 1.00}    # CBU reserve rate by category
RESERVE_RULE = "cbu"         # "cbu" = balance × CBU_MIN_RATE, for actual months and plan alike
                             # "actual" = booked reserve / balance of the last RATE_MONTHS months
                             # (either way a loan's reserve is never counted above its balance)

# History windows
MATRIX_MONTHS = 10           # migration matrix: the last N monthly transitions (10 = 01.01–01.10.2026)
RATE_MONTHS = 6              # reserve rates (only for RESERVE_RULE = "actual"): the last N months
LENDING_MONTHS = 3           # default issuance: average of the last N months
STOPPED_MONTHS = 3           # nothing issued in the last N months → the product has stopped: no issuance in the plan
MIN_LOANS = 100              # fewer loans in a matrix row → borrow the row from the department, then the currency

# Corporate: departments whose problem loans are planned loan by loan, and whose matrix never borrows retail rows
LOAN_LEVEL_DEPTS = ["kk"]
LOAN_LEVEL_FROM = 3          # loans in this category or worse on the last actual date go loan by loan
DAYS_TO_CATEGORY = {2: 31, 3: 91, 4: 181, 5: 361}   # overdue days at which a loan reaches at least this category
WRITEOFF_DAYS = 361          # old rule: written off after one year overdue
WRITEOFF_DAYS_NEW = 5 * 365 + 1   # new rule: after five years

# Scenario (1.0 = as in history)
WORSEN = 1.0                 # 1.3 = 30% more money moves to worse categories
CURE = 1.0                   # 0.8 = 20% less money comes back to better categories
LENDING_FACTOR = 1.0         # multiplies the DEFAULT lending (not the numbers you typed into the lending file)

# Write-off rule: after this date write-off takes 5 years instead of 1, for new AND existing loans. Old write-offs stay
# written off. The matrix keeps only this share of its write-off flow; the rest stays on balance in its category
WRITEOFF_CHANGE_FROM = "2027-01-01"
WRITEOFF_KEEP = 0.0          # 0 = nothing written off in 2027 (5 years is longer than the plan)
# Written-off stock (95413) stays written off until it is sold or recovered: it only shrinks, slowly
WRITEOFF_RUNOFF_MONTHS = 12  # its monthly decrease is measured over the last N months (increases are not counted)
WRITEOFF_RUNOFF = None       # None = measured; or your own share per month, e.g. 0.01 = 1% of the stock leaves monthly

# Reserve rates by hand (shares): {category: rate} for all segments, or {("UZS", "MCB", 3): 0.30} for one
RESERVE_RATE_OVERRIDE = {}

# Known events the history cannot know (a big corporate client), amounts in UNIT_NAME:
# ("01.03.2027", "UZS", "Корпоратив", from_category, to_category, amount); to_category 0 = leaves the portfolio
MANUAL_MOVES = [
    # ("01.03.2027", "UZS", "Корпоратив", 1, 3, 250_000),
]

UNIT, UNIT_NAME = 1e6, "млн сум"
BACKTEST_MONTHS = 6          # 0 = no backtest
BACKTEST_MATRIX_MONTHS = 6   # the backtest's own matrix window: keep it after the behaviour change of autumn 2025
# ═════════════════════════════════════════════════════════════════════════

In [ ]:
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from datetime import datetime

pd.options.display.float_format = "{:,.1f}".format
pd.options.display.max_columns = 40
FACT_FROM_TS, PLAN_TO_TS = pd.Timestamp(FACT_FROM), pd.Timestamp(PLAN_TO)
WO_CHANGE_TS = pd.Timestamp(WRITEOFF_CHANGE_FROM)
C = COLS


def month(s):
    """Any date → the 1st of its month."""
    return pd.to_datetime(s, errors="coerce", dayfirst=True).dt.to_period("M").dt.to_timestamp()


def to_num(s):
    """Amounts sometimes arrive as text «1 234,56»."""
    if pd.api.types.is_numeric_dtype(s):
        return s.fillna(0).astype(float)
    t = s.astype(str).str.replace(" ", "").str.replace(" ", "").str.replace(",", ".")
    return pd.to_numeric(t, errors="coerce").fillna(0.0)


def currency_group(v):
    try:
        v = str(int(float(v)))
    except (TypeError, ValueError):
        v = str(v).strip().upper()
    return "UZS" if v in UZS_CODES else "FCY"


def col_map(path):
    """Our names → the file's names, ignoring case."""
    have = {n.strip().lower(): n for n in pq.read_schema(path).names}
    return {k: have.get(v.lower()) for k, v in C.items()}


files = sorted(Path(DATA_DIR).glob("*.parquet"))
if not files:
    raise FileNotFoundError(f"no parquet files in {DATA_DIR.resolve()}")

# Which months exist: read only the date column first, so no more history is loaded than needed
all_dates = sorted(set().union(*[
    set(month(pd.read_parquet(f, columns=[col_map(f)["date"]]).iloc[:, 0]).dropna()) for f in files]))
LAST_FACT = max(all_dates)
need_from = min(FACT_FROM_TS,
                LAST_FACT - pd.DateOffset(months=BACKTEST_MONTHS + BACKTEST_MATRIX_MONTHS + 1),
                LAST_FACT - pd.DateOffset(months=max(MATRIX_MONTHS, RATE_MONTHS, LENDING_MONTHS, 12) + 1))
print(f"Files: {len(files)} · last actual date: {LAST_FACT:%d.%m.%Y} · reading from {need_from:%d.%m.%Y}")

## 1. Data
**What:** one row per loan (`anketa`) per month: segment, client, category, days overdue, balance, reserve, cumulative
write-off, issuance. If one `anketa` has several accounts, amounts are added and the **worst** category and the
largest days overdue are kept. A booked reserve is capped at the loan's balance: reserve left on a loan that is already
repaid is a booking leftover, not risk.

**Why:** migration is measured by following the *same loan* from one month to the next, so the loan must be one row.

In [ ]:
parts = []
for f in files:
    m = col_map(f)
    miss = [k for k, v in m.items() if v is None]
    if miss:
        raise KeyError(f"{f.name}: columns {[C[k] for k in miss]} not found — fix COLS")
    x = pd.read_parquet(f, columns=list(m.values()))
    x.columns = list(m.keys())
    x["date"] = month(x["date"])
    x = x[x["date"] >= need_from]
    if len(x):
        parts.append(x)
raw = pd.concat(parts, ignore_index=True)
del parts

for k in ("balance", "reserve", "written_off", "issued", "days"):
    raw[k] = to_num(raw[k])
raw["category"] = pd.to_numeric(raw["category"], errors="coerce")
bad_cat = raw["category"].isna() & (raw["balance"] > 0)
if bad_cat.any():
    print(f"⚠ {bad_cat.sum():,} rows with balance but no category → treated as category 1")
raw["category"] = raw["category"].fillna(1).clip(1, 5).astype(int)

dept = raw["department"].astype(str).str.strip().str.lower()
pas = raw["passport"].astype(str).str.strip()
raw["product"] = np.where(dept.isin(list(ONE_PRODUCT)), dept.map(ONE_PRODUCT), pas.map(lambda p: PASSPORT_GROUPS.get(p, p)))
raw["dept"] = dept
raw["cur"] = raw["currency"].map({v: currency_group(v) for v in raw["currency"].unique()})

loans = (raw.groupby(["date", "loan"], observed=True)
         .agg(cur=("cur", "first"), product=("product", "first"), dept=("dept", "first"),
              client=("client", "first"), category=("category", "max"), days=("days", "max"),
              balance=("balance", "sum"), reserve=("reserve", "sum"), written_off=("written_off", "sum"),
              issued=("issued", "sum"))
         .reset_index())
del raw
# A reserve is never more than the loan: reserve left on a repaid loan (balance 0) is ignored, and so is any excess
loans["reserve"] = loans["reserve"].clip(lower=0, upper=loans["balance"].clip(lower=0))
loans["reserve_booked"] = loans["reserve"]          # kept for the comparison in section 4
if RESERVE_RULE == "cbu":
    loans["reserve"] = loans["balance"].clip(lower=0) * loans["category"].map(CBU_MIN_RATE)
SEG = ["cur", "product"]
DEPT_OF = loans.groupby("product")["dept"].agg(lambda s: s.mode().iat[0]).to_dict()
months_all = list(pd.to_datetime(sorted(loans["date"].unique())))
print(f"Loan-months: {len(loans):,} · months: {len(months_all)} ({months_all[0]:%m.%Y}–{months_all[-1]:%m.%Y})")

# Segments = (currency, product) pairs with money in the actual window; they are the rows of the table
# (a product with only written-off money left is a row too: its «списание» line still has a stock)
size = loans[loans["date"] >= FACT_FROM_TS].groupby(SEG)[["balance", "written_off"]].sum().sum(axis=1)
size = size[size > 0]
products = size.groupby(level="product").sum().sort_values(ascending=False).index.tolist()
products = [p for p in PRODUCT_ORDER if p in products] + [p for p in products if p not in PRODUCT_ORDER]
SEGMENTS = [(c, p) for c in CURRENCY_ORDER for p in products if (c, p) in size.index]
print(f"Segments ({len(SEGMENTS)}):", ", ".join(f"{c}·{p}" for c, p in SEGMENTS))

## 2. Month-to-month transitions
**What:** for every loan with a balance in month $t$, look at the same loan in month $t+1$ and split its balance $B_t$:

| where the money went | how it is measured |
|---|---|
| **to category j** | the loan's balance $B_{t+1}$, counted in the category it has at $t+1$ |
| **written off** | growth of the cumulative write-off column, at most $B_t$ |
| **repaid** | everything else: $B_t - B_{t+1} - \text{written off}$ (a loan that disappeared is fully repaid) |

By construction, to 1…5 + repaid + written off $= B_t$ exactly. If a balance *grows* (interest, a new tranche,
FX revaluation of FCY), "repaid" comes out negative, and that is correct: the money did not leave.

**New money** $N_{t+1}$ = balance of loans that have money at $t+1$ but had none at $t$ (new loans, and old ones
coming back on balance). **Issued** = `vidan_summa` of those same loans. New money is a little smaller than issued
(some of it is repaid within the month); section 5 measures that share. A new tranche on an *existing* loan is not
new money: it shows up as negative "repaid" of that loan, so it is already inside the matrix.

**Why this split:** with these definitions nothing is lost and nothing is counted twice, so the history must be
reproduced *exactly* (check 7.1 proves it). Any error in the plan is then a forecasting error, not bookkeeping.

In [ ]:
FLOW = [f"to{j}" for j in CATEGORIES] + ["repaid", "wo"]


def transitions(t, t1):
    a = loans.loc[(loans["date"] == t) & (loans["balance"] > 0),
                  ["loan", "cur", "product", "dept", "category", "balance", "written_off"]]
    b = loans.loc[loans["date"] == t1, ["loan", "category", "balance", "written_off"]]
    m = a.merge(b, on="loan", how="left", suffixes=("", "_1"))
    present = m["balance_1"].notna()
    m["wo"] = (m["written_off_1"].fillna(m["written_off"]) - m["written_off"]).clip(lower=0)
    m["wo"] = np.minimum(m["wo"], m["balance"])
    m["balance_1"] = m["balance_1"].fillna(0)
    m["repaid"] = m["balance"] - m["balance_1"] - m["wo"]
    for j in CATEGORIES:
        m[f"to{j}"] = np.where(present & (m["category_1"] == j), m["balance_1"], 0.0)
    m["n"] = 1
    m["date"] = t1
    keep = ["date", "cur", "product", "dept", "category", "n", "balance"] + FLOW
    tr = m[keep].groupby(keep[:5], observed=True).sum().reset_index()
    # new money at t1: positive balance now, none at t; issued: what was issued to loans that had no balance at t
    had = set(a["loan"])
    fresh = loans.loc[(loans["date"] == t1) & ~loans["loan"].isin(had)]
    nb = fresh[fresh["balance"] > 0]
    nw = nb.groupby(["cur", "product", "category"], observed=True)["balance"].sum().reset_index().assign(date=t1)
    iss = fresh.groupby(["cur", "product"], observed=True)["issued"].sum().reset_index().assign(date=t1)
    return tr, nw, iss


pairs = [(a, b) for a, b in zip(months_all, months_all[1:]) if b == a + pd.DateOffset(months=1)]
_res = [transitions(a, b) for a, b in pairs]
trans = pd.concat([r[0] for r in _res], ignore_index=True)
new_money = pd.concat([r[1] for r in _res], ignore_index=True)
issued = pd.concat([r[2] for r in _res], ignore_index=True)
del _res
print(f"Monthly transitions: {len(pairs)} ({pairs[0][1]:%m.%Y}–{pairs[-1][1]:%m.%Y})")

## 3. Migration matrix $M$
**What:** for every segment and starting category $i$, add up the flows of the last `MATRIX_MONTHS` transitions and
divide by the starting balance:

$$M_{i \to j} = \frac{\sum_{\text{months}} \text{money that went from } i \text{ to } j}{\sum_{\text{months}} \text{money in } i \text{ at the start}}$$

**Why add up months first and divide once:** a quiet month and a busy month then weigh by their money, and one odd
month cannot dominate.

**Small segments:** a row built on fewer than `MIN_LOANS` loans is noise, so it is borrowed from the department
(same currency), and if that is also too small, from the whole currency. Sheet «Миграция» shows where each row came from.
**Corporate (`LOAN_LEVEL_DEPTS`) never borrows from the whole currency**, because that is retail behaviour: a corporate
row comes from corporate history only (same currency, else both currencies together), however few loans it has.

**Scenario:** `WORSEN` multiplies every move to a worse category and `CURE` every move to a better one; the difference
stays in the same category, so each row still adds up to 100%.

In [ ]:
def build_matrices(upto, months=MATRIX_MONTHS):
    """Shares per (cur, product, category) from the `months` transitions ending at `upto`."""
    lo = pd.Timestamp(upto) - pd.DateOffset(months=months)
    w = trans[(trans["date"] > lo) & (trans["date"] <= upto)]
    levels = {
        "segment": w.groupby(SEG + ["category"])[["n", "balance"] + FLOW].sum(),
        "department": w.groupby(["cur", "dept", "category"])[["n", "balance"] + FLOW].sum(),
        "department, both currencies": w.groupby(["dept", "category"])[["n", "balance"] + FLOW].sum(),
        "currency": w.groupby(["cur", "category"])[["n", "balance"] + FLOW].sum(),
    }
    rows = []
    for cur, prod in SEGMENTS:
        dep = DEPT_OF.get(prod)
        for i in CATEGORIES:
            src, r = "no history", None
            chain = [("segment", (cur, prod, i)), ("department", (cur, dep, i))]
            # corporate never borrows retail behaviour: its last resort is its own department in both currencies
            chain += ([("department, both currencies", (dep, i))] if dep in LOAN_LEVEL_DEPTS else [("currency", (cur, i))])
            found = [(name, levels[name].loc[key]) for name, key in chain
                     if key in levels[name].index and levels[name].loc[key, "balance"] > 0]
            # the most specific level with enough loans; if none has enough, the most specific one there is
            enough = [f for f in found if f[1]["n"] >= MIN_LOANS]
            if enough or found:
                src, r = (enough or found)[0]
            if r is None:      # nothing anywhere: the money stays where it is
                sh = {f: 0.0 for f in FLOW}
                sh[f"to{i}"] = 1.0
                n = 0
            else:
                sh = {f: r[f] / r["balance"] for f in FLOW}
                n = int(r["n"])
            rows.append(dict(cur=cur, product=prod, category=i, source=src, n=n, **sh))
    M = pd.DataFrame(rows).set_index(SEG + ["category"]).sort_index()
    for i in CATEGORIES:
        sel = M.index.get_level_values("category") == i
        for j in CATEGORIES:
            if j > i:
                M.loc[sel, f"to{j}"] *= WORSEN
            elif j < i:
                M.loc[sel, f"to{j}"] *= CURE
        M.loc[sel, f"to{i}"] = 1 - M.loc[sel, [f for f in FLOW if f != f"to{i}"]].sum(axis=1)
    if (M[[f"to{i}" for i in CATEGORIES]] < -1e-9).any().any():
        print("⚠ the scenario is too strong: some row has a negative 'stays' share")
    return M


M = build_matrices(LAST_FACT)
assert np.allclose(M[FLOW].sum(axis=1), 1), "a matrix row does not add up to 100%"
print("Where the matrix rows came from:", M["source"].value_counts().to_dict())


def book_matrix(upto, by=("cur",)):
    """The matrix of a whole currency (money-weighted), for a quick look."""
    lo = pd.Timestamp(upto) - pd.DateOffset(months=MATRIX_MONTHS)
    w = trans[(trans["date"] > lo) & (trans["date"] <= upto)].groupby(list(by) + ["category"])[["balance"] + FLOW].sum()
    out = w[FLOW].div(w["balance"], axis=0).mul(100)
    out.columns = [f"→{j}" for j in CATEGORIES] + ["repaid", "written off"]
    return out


book_matrix(LAST_FACT).round(1)

## 4. Reserve rates $r^i$
**What:** with `RESERVE_RULE = "cbu"` (default), $r^i$ is the CBU rate of the category (`CBU_MIN_RATE`), and the
actual months are recalculated with the same rates. With `"actual"`, $r^i = \dfrac{\sum \text{booked reserve in } i}
{\sum \text{balance in } i}$ over the last `RATE_MONTHS` months, per segment. `RESERVE_RATE_OVERRIDE` beats both.

**Why the CBU rule by default:** the reserve should depend only on where the money is. A rate measured from history
also carries one-off effects (a collateral revaluation, a loan reserved early, the general reserve booked outside the
loan data), and those would leak into the plan as noise. With a fixed rule every change in the reserve is explained by a
change in the portfolio.

**The table below** compares the rule with what was actually booked on the last date. Categories 2–5 should be close;
a big difference means collateral or something else lowers the booked reserve, and then `"actual"` may fit better.

In [ ]:
def build_rates(upto):
    lo = pd.Timestamp(upto) - pd.DateOffset(months=RATE_MONTHS)
    w = loans[(loans["date"] > lo) & (loans["date"] <= upto) & (loans["balance"] > 0)]
    lv = [w.groupby(SEG + ["category"])[["reserve", "balance"]].sum(),
          w.groupby(["cur", "dept", "category"])[["reserve", "balance"]].sum(),
          w.groupby(["cur", "category"])[["reserve", "balance"]].sum()]
    out = {}
    for cur, prod in SEGMENTS:
        for i in CATEGORIES:
            rate = CBU_MIN_RATE[i]
            for g, key in zip(lv, ((cur, prod, i), (cur, DEPT_OF.get(prod), i), (cur, i))):
                if key in g.index and g.loc[key, "balance"] > 0:
                    rate = g.loc[key, "reserve"] / g.loc[key, "balance"]
                    break
            out[(cur, prod, i)] = RESERVE_RATE_OVERRIDE.get((cur, prod, i), RESERVE_RATE_OVERRIDE.get(i, rate))
    return pd.Series(out, name="rate").rename_axis(SEG + ["category"])


R = build_rates(LAST_FACT)
last = loans[(loans["date"] == LAST_FACT) & (loans["balance"] > 0)].groupby(["cur", "category"])[
    ["balance", "reserve_booked", "reserve"]].sum()
rate_cmp = pd.DataFrame({"booked, %": last["reserve_booked"] / last["balance"] * 100,
                         "used in the plan, %": last["reserve"] / last["balance"] * 100,
                         f"booked, {UNIT_NAME}": last["reserve_booked"] / UNIT,
                         f"used, {UNIT_NAME}": last["reserve"] / UNIT})
print(f"Reserve rule: {RESERVE_RULE} · booked vs used on {LAST_FACT:%d.%m.%Y}:")
rate_cmp.round(1)

## 5. Issuance and new money $N_t$
**What:**
1. **The last 12 months of issuance** (`vidan_summa` of new loans) per segment are shown first, so stopped and
   growing products are visible.
2. **Default issuance** = the average of the last `LENDING_MONTHS` months × `LENDING_FACTOR`. If a segment issued
   **nothing in the last `STOPPED_MONTHS` months**, it has stopped lending and gets **0** for the whole plan.
3. It is written to `reserve_plan_vidacha.xlsx` (rows = segments, columns = plan months, in `UNIT_NAME`). Change it
   when there is a business plan and rerun; empty cells fall back to the default.
4. Issuance becomes new money: $N = \text{issued} \times k$, where
   $k = \dfrac{\sum \text{new money}}{\sum \text{issued}}$ over the last `LENDING_MONTHS` months, per segment
   (the share still on balance at the end of the month). New money is split into categories the way it was recently.

**Why:** issuance is a business decision, not a statistic, and a business plan is written in issuance, so that is
what you edit. The share $k$ turns it into the money that actually lands on the balance sheet.

In [ ]:
plan_dates = list(pd.date_range(LAST_FACT + pd.DateOffset(months=1), PLAN_TO_TS, freq="MS"))
trans_dates = sorted(issued["date"].unique())


def recent(df, n):
    """Rows of the last n monthly transitions."""
    return df[df["date"].isin(trans_dates[-n:])]


iss_seg = issued.groupby(SEG + ["date"])["issued"].sum()
last12 = (iss_seg[iss_seg.index.get_level_values("date").isin(trans_dates[-12:])]
          .unstack("date").reindex(pd.MultiIndex.from_tuples(SEGMENTS, names=SEG)).fillna(0))
avg_iss = recent(issued, LENDING_MONTHS).groupby(SEG)["issued"].sum() / LENDING_MONTHS
stopped = [s for s in SEGMENTS if recent(issued, STOPPED_MONTHS).groupby(SEG)["issued"].sum().get(s, 0) <= 0]
nm_recent = recent(new_money, LENDING_MONTHS).groupby(SEG)["balance"].sum()
iss_recent = recent(issued, LENDING_MONTHS).groupby(SEG)["issued"].sum()
K = {s: float(np.clip(nm_recent.get(s, 0) / iss_recent[s], 0, 1.5)) if iss_recent.get(s, 0) > 0 else 1.0
     for s in SEGMENTS}
_mix = recent(new_money, max(LENDING_MONTHS, 6)).groupby(SEG + ["category"])["balance"].sum()
MIX = (_mix / _mix.groupby(level=SEG).transform("sum")).to_dict()


def mix_of(cur, prod):
    m = np.array([MIX.get((cur, prod, i), 0.0) for i in CATEGORIES])
    return m if m.sum() > 0 else np.array([float(i == 1) for i in CATEGORIES])


default_iss = {s: 0.0 if s in stopped else avg_iss.get(s, 0.0) * LENDING_FACTOR for s in SEGMENTS}
default_lending = pd.DataFrame(
    [[cur, prod] + [default_iss[(cur, prod)] / UNIT] * len(plan_dates) for cur, prod in SEGMENTS],
    columns=["Валюта", "Продукт"] + [d.strftime("%d.%m.%Y") for d in plan_dates])

if LENDING_FILE.exists():
    user = pd.read_excel(LENDING_FILE)
    user.columns = [c.strftime("%d.%m.%Y") if hasattr(c, "strftime") else str(c).strip() for c in user.columns]
    lend_tbl = default_lending[["Валюта", "Продукт"]].merge(user, on=["Валюта", "Продукт"], how="left")
    for c in default_lending.columns[2:]:
        lend_tbl[c] = lend_tbl[c].fillna(default_lending[c]) if c in lend_tbl else default_lending[c]
    lend_tbl = lend_tbl[default_lending.columns]
    print(f"Issuance taken from {LENDING_FILE} (empty cells, new segments and new months = the default)")
else:
    lend_tbl = default_lending
    lend_tbl.to_excel(LENDING_FILE, index=False)
    print(f"Created {LENDING_FILE}: average of {LENDING_MONTHS} months, stopped products = 0. Edit it and rerun.")

LENDING = {(r["Валюта"], r["Продукт"], d): float(r[d.strftime("%d.%m.%Y")]) * UNIT
           for _, r in lend_tbl.iterrows() for d in plan_dates}
if stopped:
    print(f"Stopped (nothing issued in the last {STOPPED_MONTHS} months): " + ", ".join(f"{c}·{p}" for c, p in stopped))

iss_view = last12.div(UNIT)
iss_view.columns = [pd.Timestamp(d).strftime("%m.%y") for d in iss_view.columns]
iss_view[f"avg {LENDING_MONTHS} m"] = [avg_iss.get(s, 0.0) / UNIT for s in iss_view.index]
iss_view["k, %"] = [K[s] * 100 for s in iss_view.index]
iss_view["status"] = ["stopped" if s in stopped else "" for s in iss_view.index]
print(f"Issuance by month, {UNIT_NAME} (the month is the report date the issuance arrives on):")
iss_view.round(1)

## 6. The projection
**What:** starting from the actual portfolio on the last date, apply $S_{t+1} = S_t \cdot M + N_{t+1}$ month after month
to `PLAN_TO`, and $\text{Reserve} = \sum S^i r^i$ for every month. The plan has two parts that are added together:

**Part 1, the matrix:** every loan except the problem corporate loans.

**Part 2, loan by loan:** corporate loans (`LOAN_LEVEL_DEPTS`) in category `LOAN_LEVEL_FROM` or worse on the last
actual date. Each one, every month:
* **days overdue keep the pace of the last actual month**: if they grew by a full month (nothing paid), they keep
  growing a full month every month; if they stood still (the client pays), they stay. Pace = growth of `maks_dni`
  in the last month ÷ days in that month, between 0 and 1;
* **the category follows the days** (`DAYS_TO_CATEGORY`, e.g. 361+ days → category 5) and never improves by itself;
* **it is written off** in the month its days cross the write-off term: one year (`WRITEOFF_DAYS`) under the old rule,
  five years (`WRITEOFF_DAYS_NEW`) after `WRITEOFF_CHANGE_FROM`, so in 2027 it stays on balance in category 5.
  A loan that is *already* past the term and still on balance is being kept by the bank (court, collateral…), so it
  is not written off automatically; it stays in category 5 unless you decide otherwise;
* **your decision replaces all of this** from its date: in `reserve_plan_corporate.xlsx` write «погашен», «списан» or
  «категория» (+ the new category) and the date. Until that date the loan follows its days but is not written off
  automatically; after it, the loan stays as decided.

**Why loan by loan:** a loan cannot be 13% written off and 20% cured; it moves as one piece. With a handful of big loans
an average is wrong for every single one of them, while the risk team knows each one by name.

**The write-off rule change:** after `WRITEOFF_CHANGE_FROM` the write-off term is five years for new **and** existing
loans, so nothing is written off in 2027. In the matrix, the "written off" column of $M$ is multiplied by
`WRITEOFF_KEEP` (0) and that money **stays in its own category**; loan by loan, the term simply becomes five years.
Old write-offs stay written off: the model never brings them back.

**The written-off stock (account 95413)** is shown as the «списание» row of the portfolio table. It does not cure;
it only leaves when it is sold or recovered. Its plan: stock(t) = stock(t−1) × (1 − $d$) + new write-offs of the plan,
where $d$ = Σ monthly *decreases* ÷ Σ stock over the last `WRITEOFF_RUNOFF_MONTHS` months, per product (else the whole
currency). Months where a loan's written-off amount grew are not counted as decrease: that was new write-off, and from
2027 there is none. So the stock grows a little until 2027 (the last write-offs) and then slowly melts.
`WRITEOFF_RUNOFF` replaces $d$ with your own number (e.g. a sale plan).
(Report date 01.01.2027 is the portfolio at the end of December 2026, so December's flow still follows the old rule;
the new rule starts with 01.02.2027 = January's flow.)

`MANUAL_MOVES` are applied in their month, after the matrix.

In [ ]:
def stock_at(d):
    return loans[(loans["date"] == d) & (loans["balance"] > 0)].groupby(SEG + ["category"])[["balance", "reserve"]].sum()


def cat_by_days(days):
    """The category that `days` overdue reach at least."""
    return max([1] + [c for c, lim in DAYS_TO_CATEGORY.items() if days >= lim])


def loan_key(x):
    s = str(x).strip()
    return s[:-2] if s.endswith(".0") else s


def problem_loans(d):
    """Corporate loans in category LOAN_LEVEL_FROM or worse on date d: these are planned loan by loan.
    pace = how fast their days overdue grew in the last month (1 = a full month, nothing paid; 0 = stood still)."""
    x = loans[(loans["date"] == d) & (loans["balance"] > 0) & loans["dept"].isin(LOAN_LEVEL_DEPTS)
              & (loans["category"] >= LOAN_LEVEL_FROM)]
    prev = d - pd.DateOffset(months=1)
    before = loans[loans["date"] == prev].set_index("loan")["days"]
    grew = x["days"].to_numpy() - x["loan"].map(before).to_numpy()
    x = x.assign(pace=np.nan_to_num(np.clip(grew / (d - prev).days, 0, 1), nan=1.0))
    return x[["loan", "client", "cur", "product", "category", "days", "pace", "balance"]].reset_index(drop=True)


def loan_path(L, start, dates, wo_change, decision=None):
    """One loan month by month → [(date, category, balance, repaid, written_off)].
    decision = (date, "repaid" | "written off" | "category", new_category) replaces the automatic path from its date."""
    cat, days, bal, prev, decided = int(L.category), float(L.days), float(L.balance), start, False
    out = []
    for d in dates:
        repaid = wo = 0.0
        if bal > 0 and not decided and decision is not None and d >= decision[0]:
            decided = True
            if decision[1] == "repaid":
                repaid, bal = bal, 0.0
            elif decision[1] == "written off":
                wo, bal = bal, 0.0
            else:
                cat = int(decision[2])
        elif bal > 0 and not decided:
            before, days = days, days + L.pace * (d - prev).days
            cat = max(cat, cat_by_days(days))
            term = WRITEOFF_DAYS if d <= wo_change else WRITEOFF_DAYS_NEW
            # written off in the month it crosses the term; already past it = the bank keeps it; and a loan with
            # your decision waits for that decision instead
            if before < term <= days and decision is None:
                wo, bal = bal, 0.0
        prev = d
        out.append((d, cat, bal, repaid, wo))
    return out


def path_text(path, cat0):
    parts, now = [], cat0
    for d, cat, bal, rep, wo in path:
        if wo > 0:
            parts.append(f"списан {d:%d.%m.%Y}")
            break
        if cat != now:
            parts.append(f"кат. {cat} с {d:%d.%m.%Y}")
            now = cat
    return " → ".join(parts) or f"остаётся в кат. {cat0}"


# The corporate file: created on the first run; on later runs your decisions are read from it
ACTIONS = {"погаш": "repaid", "спис": "written off", "катег": "category"}
held_now = problem_loans(LAST_FACT)
if not CORPORATE_FILE.exists():
    cf = pd.DataFrame({
        "Валюта": held_now["cur"], "Продукт": held_now["product"], "Клиент": held_now["client"],
        "Анкета": held_now["loan"].map(loan_key), f"Остаток, {UNIT_NAME}": (held_now["balance"] / UNIT).round(1),
        "Категория": held_now["category"], "Дней просрочки": held_now["days"].astype(int),
        "Темп просрочки, %": (held_now["pace"] * 100).round(0),
        "Авто: путь": [path_text(loan_path(L, LAST_FACT, plan_dates, WO_CHANGE_TS), int(L.category))
                       for L in held_now.itertuples()],
        "Решение": "", "Дата решения": "", "Новая категория": ""})
    cf.sort_values(["Валюта", "Клиент", "Анкета"]).to_excel(CORPORATE_FILE, index=False)
    print(f"Created {CORPORATE_FILE}: {len(cf)} problem corporate loans. Write your decisions in it and rerun.")
cf = pd.read_excel(CORPORATE_FILE, dtype={"Анкета": str})
DECISIONS = {}
for _, r in cf.iterrows():
    act = str(r.get("Решение", "")).strip().lower()
    if act in ("", "nan"):
        continue
    kind = next((v for k, v in ACTIONS.items() if act.startswith(k)), None)
    when = pd.to_datetime(r.get("Дата решения"), dayfirst=True, errors="coerce")
    new_cat = pd.to_numeric(r.get("Новая категория"), errors="coerce")
    if kind is None or pd.isna(when) or (kind == "category" and new_cat not in CATEGORIES):
        print(f"⚠ loan {r['Анкета']}: decision «{r.get('Решение')}» / «{r.get('Дата решения')}» / "
              f"«{r.get('Новая категория')}» not understood — automatic path used")
        continue
    DECISIONS[loan_key(r["Анкета"])] = (when, kind, new_cat)
missing = set(held_now["loan"].map(loan_key)) - set(cf["Анкета"].map(loan_key))
if missing:
    print(f"⚠ {len(missing)} problem loans are not in {CORPORATE_FILE} (new since it was made); they follow the "
          f"automatic path. Rename the file to have it made again.")
print(f"Problem corporate loans on {LAST_FACT:%d.%m.%Y}: {len(held_now)}, "
      f"{held_now['balance'].sum() / UNIT:,.1f} {UNIT_NAME}; decisions read: {len(DECISIONS)}")
# DAYS_TO_CATEGORY must match how your loans are actually classified: days overdue by category, whole book, last date
days_chk = (loans[(loans["date"] == LAST_FACT) & (loans["balance"] > 0)].groupby("category")["days"]
            .describe(percentiles=[.05, .5, .95])[["count", "min", "5%", "50%", "95%", "max"]])
print("Days overdue by category on the last date (compare with DAYS_TO_CATEGORY):")
display(days_chk.round(0))
cf

In [ ]:
def project(start, dates, Mx, Rx, new_by_cat, wo_change=WO_CHANGE_TS, wo_keep=WRITEOFF_KEEP, moves=(), decisions=None):
    """Portfolio by category for every date in `dates`, from the actual portfolio at `start`.
    new_by_cat(cur, prod, d) → array of 5: new money landing in categories 1..5."""
    held = problem_loans(start)
    s0 = stock_at(start)["balance"].sub(held.groupby(SEG + ["category"])["balance"].sum(), fill_value=0)
    rows, flows = [], []
    # part 1: the matrix
    for cur, prod in SEGMENTS:
        v = np.array([s0.get((cur, prod, i), 0.0) for i in CATEGORIES])
        P = Mx.loc[(cur, prod)][FLOW].to_numpy()                          # 5 rows × 7 columns
        for d in dates:
            Pd = P.copy()
            if d > wo_change:                                             # write-offs that no longer happen stay put
                kept = Pd[:, -1] * (1 - wo_keep)
                Pd[:, -1] -= kept
                Pd[np.arange(5), np.arange(5)] += kept
            flow = v @ Pd                                                 # → 1..5, repaid, written off
            new = new_by_cat(cur, prod, d)
            v = flow[:5] + new
            for md, mc, mp, fr, to, amt in moves:
                if (pd.Timestamp(datetime.strptime(md, "%d.%m.%Y")), mc, mp) == (d, cur, prod):
                    amt = min(amt * UNIT, v[fr - 1])                      # never move more than is there
                    v[fr - 1] -= amt
                    if to:
                        v[to - 1] += amt
            flows.append(dict(date=d, cur=cur, product=prod, new_money=new.sum(), repaid=flow[5], written_off=flow[6]))
            for i, b in zip(CATEGORIES, v):
                rows.append(dict(date=d, cur=cur, product=prod, category=i, balance=b))
    # part 2: problem corporate loans, one by one
    for L in held.itertuples():
        for d, cat, bal, rep, wo in loan_path(L, start, dates, wo_change, (decisions or {}).get(loan_key(L.loan))):
            rows.append(dict(date=d, cur=L.cur, product=L.product, category=cat, balance=bal))
            flows.append(dict(date=d, cur=L.cur, product=L.product, new_money=0.0, repaid=rep, written_off=wo))
    out = pd.DataFrame(rows).groupby(["date", "cur", "product", "category"], as_index=False)["balance"].sum()
    out["reserve"] = [b * Rx.get((c, p, i), CBU_MIN_RATE[i])
                      for c, p, i, b in zip(out["cur"], out["product"], out["category"], out["balance"])]
    return out, pd.DataFrame(flows).groupby(["date", "cur", "product"], as_index=False).sum()


def planned_new(c, p, d):
    """Planned issuance × share that lands on balance × category mix."""
    return LENDING.get((c, p, d), 0.0) * K.get((c, p), 1.0) * mix_of(c, p)


plan, plan_flows = project(LAST_FACT, plan_dates, M, R, planned_new, moves=MANUAL_MOVES, decisions=DECISIONS)

fact_dates = [d for d in months_all if d >= FACT_FROM_TS]
fact = pd.concat([stock_at(d).reset_index().assign(date=d) for d in fact_dates], ignore_index=True)
both = pd.concat([fact.assign(kind="fact"), plan.assign(kind="plan")], ignore_index=True)

# Written off during the month (a flow): actual = growth of the write-off column (section 2), plan = the projection
wo_fact = trans[trans["date"].isin(fact_dates)].groupby(["date"] + SEG)["wo"].sum().rename("written_off").reset_index()
wo_flow = pd.concat([wo_fact.assign(kind="fact"), plan_flows[["date"] + SEG + ["written_off"]].assign(kind="plan")],
                    ignore_index=True)

# Written-off STOCK (95413): actual = the write-off column of all loans on the date. Its monthly decrease (sold,
# recovered) is measured loan by loan; a loan whose written-off amount went up is not counted, since from 2027
# nothing new is written off. Plan: stock(t) = stock(t-1) × (1 − decrease rate) + new write-offs of the plan.
wo_rows = loans.loc[loans["written_off"] > 0, ["date", "loan", "cur", "product", "written_off"]]
wo_hist = wo_rows.groupby(["date"] + SEG)["written_off"].sum()
_dec = []
for t, t1 in pairs[-WRITEOFF_RUNOFF_MONTHS:]:
    a = wo_rows[wo_rows["date"] == t]
    later = wo_rows.loc[wo_rows["date"] == t1].set_index("loan")["written_off"]
    gone = (a["written_off"] - a["loan"].map(later).fillna(0)).clip(lower=0)      # decreases only
    _dec.append(a.assign(gone=gone).groupby(SEG)[["written_off", "gone"]].sum())
_dec = pd.concat(_dec) if _dec else pd.DataFrame(columns=["written_off", "gone"])
d_seg, d_cur = _dec.groupby(level=SEG).sum(), _dec.groupby(level="cur").sum()


def runoff_rate(cur, prod):
    if WRITEOFF_RUNOFF is not None:
        return WRITEOFF_RUNOFF
    for g, key in ((d_seg, (cur, prod)), (d_cur, cur)):          # the segment, else the whole currency
        if key in g.index and g.loc[key, "written_off"] > 0:
            return g.loc[key, "gone"] / g.loc[key, "written_off"]
    return 0.0


RUNOFF = {s: runoff_rate(*s) for s in SEGMENTS}
_new_wo = plan_flows.set_index(["date"] + SEG)["written_off"]
_rows = []
for cur, prod in SEGMENTS:
    st = wo_hist.get((LAST_FACT, cur, prod), 0.0)
    for d in plan_dates:
        st = st * (1 - RUNOFF[(cur, prod)]) + _new_wo.get((d, cur, prod), 0.0)
        _rows.append(dict(date=d, cur=cur, product=prod, stock=st))
writeoffs = pd.concat([
    wo_hist[wo_hist.index.get_level_values("date").isin(fact_dates)].rename("stock").reset_index().assign(kind="fact"),
    pd.DataFrame(_rows).assign(kind="plan")], ignore_index=True)

wo_view = pd.DataFrame({
    f"stock {LAST_FACT:%d.%m.%Y}, {UNIT_NAME}": [wo_hist.get((LAST_FACT, c, p), 0.0) / UNIT for c, p in SEGMENTS],
    "decrease, % a month": [RUNOFF[s] * 100 for s in SEGMENTS],
    f"stock {plan_dates[-1]:%d.%m.%Y}, {UNIT_NAME}": [
        writeoffs[(writeoffs["date"] == plan_dates[-1]) & (writeoffs["cur"] == c) & (writeoffs["product"] == p)]["stock"].sum() / UNIT
        for c, p in SEGMENTS]}, index=pd.MultiIndex.from_tuples(SEGMENTS, names=SEG))
wo_view = wo_view[wo_view.iloc[:, 0] + wo_view.iloc[:, 2] > 0]
print(f"Written-off stock (95413), decrease measured over {WRITEOFF_RUNOFF_MONTHS} months"
      + ("" if WRITEOFF_RUNOFF is None else f" — overridden: {WRITEOFF_RUNOFF:.1%} a month") + ":")
display(wo_view.round(1))

tot = both.groupby(["date", "kind"])[["balance", "reserve"]].sum().div(UNIT)
npl = both[both["category"].isin(NPL_CATEGORIES)].groupby("date")["balance"].sum().div(UNIT)
tot["NPL"] = npl.reindex(tot.index.get_level_values("date")).to_numpy()
tot["NPL %"] = tot["NPL"] / tot["balance"] * 100
tot["reserve / portfolio %"] = tot["reserve"] / tot["balance"] * 100
tot["written off in the month"] = wo_flow.groupby(["date", "kind"])["written_off"].sum().div(UNIT)
tot["written-off stock (95413)"] = writeoffs.groupby(["date", "kind"])["stock"].sum().div(UNIT)
tot.rename(columns={"balance": f"portfolio, {UNIT_NAME}", "reserve": f"reserve, {UNIT_NAME}"})

### 6.1 What the write-off rule change does
**What:** the same plan run a second time with the **old** rule (write-off continues as in history), and the
difference at the end of the plan.

**Why:** the rule change is the one thing in 2027 that history has never seen, so its effect should be a number of
its own. If it is ≈ 0, check how much is written off per month today (7.4, «4 written off»).

In [ ]:
plan_old, flows_old = project(LAST_FACT, plan_dates, M, R, planned_new,
                              wo_change=pd.Timestamp.max, moves=MANUAL_MOVES, decisions=DECISIONS)
end = plan_dates[-1]
eff = pd.concat({"old rule": plan_old[plan_old["date"] == end].groupby("cur")[["balance", "reserve"]].sum(),
                 "new rule": plan[plan["date"] == end].groupby("cur")[["balance", "reserve"]].sum()}, axis=1).div(UNIT)
for k in ("balance", "reserve"):
    eff[("difference", k)] = eff[("new rule", k)] - eff[("old rule", k)]
eff[("not written off", "in the plan")] = (flows_old.groupby("cur")["written_off"].sum()
                                           - plan_flows.groupby("cur")["written_off"].sum()) / UNIT
print(f"6.1 On {end:%d.%m.%Y}, {UNIT_NAME}:")
eff.round(1)

## 7. Checks: can this plan be trusted?
Three checks, from the most basic to the hardest. Read them in this order: each one only matters if the one before passed.

### 7.1 Bookkeeping: does the machinery reproduce history exactly?
**What:** for every past month, rebuild $S_{t+1}$ from $S_t$ using *that month's own* flows plus that month's new money,
and compare with the actual $S_{t+1}$.

**Why:** this involves no forecasting at all, so the difference must be ≈ 0. If it is not, the data has a problem
(loans changing currency, duplicate rows, a month missing) and every number after this is suspect.

In [ ]:
arrived = trans.groupby(["date", "cur"])[[f"to{j}" for j in CATEGORIES]].sum()
arrived.columns = CATEGORIES
arrived = arrived.stack().rename("arrived")
arrived.index.names = ["date", "cur", "category"]
entered = new_money.groupby(["date", "cur", "category"])["balance"].sum().rename("new")
actual = loans[loans["balance"] > 0].groupby(["date", "cur", "category"])["balance"].sum().rename("actual")
ident = pd.concat([arrived, entered], axis=1).fillna(0).join(actual, how="outer").fillna(0)
ident = ident[ident.index.get_level_values("date").isin([b for _, b in pairs])]
ident["rebuilt"] = ident["arrived"] + ident["new"]
gap = (ident["rebuilt"] - ident["actual"]).abs().groupby(level="date").sum() / ident["actual"].groupby(level="date").sum() * 100
print(f"7.1 Bookkeeping: largest monthly gap = {gap.max():.4f}% of the portfolio "
      + ("✓ (history is reproduced exactly)" if gap.max() < 0.01 else "⚠ look at the months below"))
gap[gap > 0.01].round(3)

### 7.2 One month ahead: is the matrix right for next month?
**What:** for each of the last `BACKTEST_MONTHS` months, take the actual $S_t$, apply the plan's machinery (matrix built
only from the `BACKTEST_MATRIX_MONTHS` before it, problem corporate loans loan by loan), add the *actual* new money of
month $t+1$, and compare with the actual $S_{t+1}$.

**Why:** this isolates the matrix. If one month ahead is already wrong, the matrix misses something about the
portfolio. If one month ahead is fine but twelve months ahead (7.3) is wrong, small errors are adding up month after
month, or the portfolio's behaviour changed during the year (look at 7.4).

In [ ]:
one = []
nm_all = new_money.set_index(["cur", "product", "date", "category"])["balance"].to_dict()


def actual_new(c, p, d):
    return np.array([nm_all.get((c, p, d, i), 0.0) for i in CATEGORIES])


for t, t1 in pairs[-BACKTEST_MONTHS:] if BACKTEST_MONTHS else []:
    p1, _ = project(t, [t1], build_matrices(t, BACKTEST_MATRIX_MONTHS), R, actual_new, wo_change=pd.Timestamp.max)
    act = stock_at(t1)["balance"]
    one += [dict(date=t1, cur=r.cur, product=r.product, category=r.category, forecast=r.balance,
                 actual=act.get((r.cur, r.product, r.category), 0.0)) for r in p1.itertuples()]
one = pd.DataFrame(one)
if len(one):
    o = one.groupby(["date", "category"])[["forecast", "actual"]].sum()
    one_err = (o["forecast"] / o["actual"] - 1).mul(100).unstack("category")
    one_err.index = one_err.index.strftime("%m.%Y")
    print("7.2 One month ahead, error by category, % (forecast / actual − 1):")
    display(one_err.round(1))

### 7.3 Backtest: several months ahead, as if it were then
**What:** stand on the date `BACKTEST_MONTHS` months back, build the matrix (`BACKTEST_MATRIX_MONTHS`) and the rates
only from what was known then, plan the problem corporate loans of that date loan by loan (automatic path),
feed in the **actual** new money of each month (so the test is about migration, not lending), apply the old write-off
rule, and run to the last actual date. Compare by category, by product, and month by month.

**Why:** this is exactly what the 2027 plan does, tested on a year whose answer we know.

**Why its own window `BACKTEST_MATRIX_MONTHS`:** 7.4 shows the behaviour changed in autumn 2025. A backtest whose
matrix reaches back before that is built on the *old* behaviour and must miss, which says nothing about the plan.
With 6 + 6 months both windows sit after the change.

In [ ]:
bt = bt_prod = bt_path = None
if BACKTEST_MONTHS:
    bt_start = LAST_FACT - pd.DateOffset(months=BACKTEST_MONTHS)
    if bt_start in set(months_all):
        bt_dates = list(pd.date_range(bt_start + pd.DateOffset(months=1), LAST_FACT, freq="MS"))
        bt_plan, _ = project(bt_start, bt_dates, build_matrices(bt_start, BACKTEST_MATRIX_MONTHS),
                             build_rates(bt_start), actual_new, wo_change=pd.Timestamp.max)
        f_all = pd.concat([stock_at(d).reset_index().assign(date=d) for d in bt_dates], ignore_index=True)

        def compare(by):
            p_ = bt_plan[bt_plan["date"] == LAST_FACT].groupby(by)[["balance", "reserve"]].sum()
            f_ = f_all[f_all["date"] == LAST_FACT].groupby(by)[["balance", "reserve"]].sum()
            out = pd.concat({"forecast": p_, "actual": f_}, axis=1).fillna(0).div(UNIT)
            for k in ("balance", "reserve"):
                out[("error %", k)] = (out[("forecast", k)] / out[("actual", k)].replace(0, np.nan) - 1) * 100
            return out

        bt = compare(["cur", "category"])
        bt_prod = compare(["cur", "product"])
        bt_prod[("gap", "balance")] = bt_prod[("forecast", "balance")] - bt_prod[("actual", "balance")]
        bt_prod = bt_prod.sort_values(("gap", "balance"), key=abs, ascending=False)
        pp = bt_plan.groupby(["date", "category"])["balance"].sum()
        fp = f_all.groupby(["date", "category"])["balance"].sum()
        bt_path = (pp / fp - 1).mul(100).unstack("category")
        bt_path.index = bt_path.index.strftime("%m.%Y")
        tp, tf = bt_plan[bt_plan["date"] == LAST_FACT][["balance", "reserve"]].sum(), f_all[f_all["date"] == LAST_FACT][["balance", "reserve"]].sum()
        print(f"7.3 Backtest {bt_start:%d.%m.%Y} → {LAST_FACT:%d.%m.%Y}: "
              f"portfolio {tp['balance'] / tf['balance'] * 100 - 100:+.1f}%, reserve {tp['reserve'] / tf['reserve'] * 100 - 100:+.1f}%")
        print("\nBy category:")
        display(bt.round(1))
        print("By product — biggest money gaps first (this is where to look):")
        display(bt_prod.head(15).round(1))
        print("Month by month, error by category, % — does the error appear at once or grow?")
        display(bt_path.round(1))
    else:
        print(f"Backtest skipped: no data on {bt_start:%d.%m.%Y}")

### 7.4 Did the behaviour change? The key flows month by month
**What:** the most important cells of the whole-currency matrix, measured every single month (money-weighted).

**Why:** the matrix is an average of the last `MATRIX_MONTHS`. If, say, 1→2 was 4% a month in 2025 and 2% in 2026,
a matrix built on 2025 overstates 2026, and the backtest error is explained by a real change in the portfolio, not
by the method. If the flows are drifting, use a shorter `MATRIX_MONTHS` (e.g. 6) so the plan follows today's behaviour.

In [ ]:
key = trans.groupby(["date", "cur", "category"])[["balance"] + FLOW].sum()
kf = pd.DataFrame(index=key.index.droplevel("category").unique())
for i, label, cols in [(1, "1→2", ["to2"]), (1, "1→3,4,5", ["to3", "to4", "to5"]), (1, "1 repaid", ["repaid"]),
                       (2, "2→1", ["to1"]), (2, "2→3,4,5", ["to3", "to4", "to5"]),
                       (3, "3→1,2", ["to1", "to2"]), (3, "3→4,5", ["to4", "to5"]),
                       (4, "4→1,2,3", ["to1", "to2", "to3"]), (4, "4 written off", ["wo"])]:
    k = key.xs(i, level="category")
    kf[label] = (k[cols].sum(axis=1) / k["balance"] * 100).reindex(kf.index)
kf = kf.reset_index()
kf["date"] = kf["date"].dt.strftime("%m.%Y")
print("7.4 Monthly flows, % of the money in the starting category:")
for cur in CURRENCY_ORDER:
    print(cur)
    display(kf[kf["cur"] == cur].drop(columns="cur").set_index("date").round(1))

## 8. Charts

In [ ]:
import matplotlib.pyplot as plt

KB = {1: "#F2C230", 2: "#C9971A", 3: "#8A867C", 4: "#5E5A52", 5: "#C0392B"}   # red = the worst only
for cur in CURRENCY_ORDER:
    g = both[both["cur"] == cur].groupby(["date", "category"])[["balance", "reserve"]].sum().div(UNIT)
    if g.empty:
        continue
    bal = g["balance"].unstack("category").reindex(columns=CATEGORIES, fill_value=0)
    res = g["reserve"].groupby(level="date").sum()
    fig, (ax, ax2) = plt.subplots(2, 1, figsize=(13, 7.5), sharex=True, gridspec_kw={"height_ratios": [3, 2]})
    x = np.arange(len(bal))
    bottom = np.zeros(len(bal))
    for i in CATEGORIES:
        ax.bar(x, bal[i], bottom=bottom, color=KB[i], label=f"cat. {i}", width=0.8)
        bottom += bal[i].to_numpy()
    ax2.plot(x, res.to_numpy(), color="#2A2A2E", marker="o", lw=2)     # own scale: the reserve is far smaller
    split = sum(d <= LAST_FACT for d in bal.index) - 0.5
    for a in (ax, ax2):
        a.axvline(split, color="#2A2A2E", ls="--", lw=1)
        a.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:,.0f}".replace(",", " ")))
        a.spines[["top", "right"]].set_visible(False)
    ax.text(split + 0.2, ax.get_ylim()[1] * 0.97, "plan →", va="top")
    ax.set_title(f"{cur}: portfolio by category, {UNIT_NAME}", loc="left")
    ax2.set_title(f"{cur}: CBU reserve, {UNIT_NAME}", loc="left")
    ax.legend(ncol=5, loc="lower right", bbox_to_anchor=(1, 1.0), frameon=False)
    ax2.set_xticks(x, [d.strftime("%m.%y") for d in bal.index], rotation=90)
    plt.tight_layout()
    plt.show()

## 9. Excel
Sheet «План» is the template: Валюта · Продукт · Категория · months (actual, then plan in yellow), portfolio on top,
reserve below. The other sheets show everything the plan was built from, so every number can be traced.

In [ ]:
from openpyxl import Workbook
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter

dates_all = sorted(both["date"].unique())
piv = both.pivot_table(index=SEG + ["category"], columns="date", values=["balance", "reserve"], aggfunc="sum").fillna(0)

wb = Workbook()
ws = wb.active
ws.title = "План"
BOLD, HEAD = Font(bold=True), PatternFill("solid", fgColor="2A2A2E")
PLAN_FILL, TOTAL_FILL = PatternFill("solid", fgColor="FFF4CC"), PatternFill("solid", fgColor="EDEBE4")
NUM = "#,##0.0;-#,##0.0;-"
ws.column_dimensions["A"].width, ws.column_dimensions["B"].width, ws.column_dimensions["C"].width = 9, 24, 10


wo_piv = (writeoffs.pivot_table(index=SEG, columns="date", values="stock", aggfunc="sum")
          .reindex(columns=dates_all).fillna(0))
WO_LABEL = "списание"


def block(start_row, value, title, with_wo=False):
    """One table of the template; with_wo adds the month's write-off under categories 1-5 of every product."""
    r = start_row
    ws.cell(r, 1, f"{title}, {UNIT_NAME}").font = Font(bold=True, size=13)
    r += 1
    heads = ["Валюта", "Продукт", "Категория"] + [pd.Timestamp(d).strftime("%d.%m.%Y") for d in dates_all]
    for c, h in enumerate(heads, 1):
        cell = ws.cell(r, c, h)
        cell.font, cell.fill, cell.alignment = Font(bold=True, color="FFFFFF"), HEAD, Alignment(horizontal="center")
    r += 1
    for c, d in enumerate(dates_all, 4):
        ws.cell(r, c, "план" if pd.Timestamp(d) > LAST_FACT else "факт").alignment = Alignment(horizontal="center")
    r += 1
    first = r
    pv = piv[value].reindex(pd.MultiIndex.from_tuples([(c, p, i) for c, p in SEGMENTS for i in CATEGORIES]), fill_value=0)
    for cur, prod in SEGMENTS:
        for i in CATEGORIES:
            ws.cell(r, 1, cur), ws.cell(r, 2, prod), ws.cell(r, 3, i)
            vals = pv.loc[(cur, prod, i)]
            for c, d in enumerate(dates_all, 4):
                cell = ws.cell(r, c, round(float(vals.iloc[c - 4]) / UNIT, 3))
                cell.number_format = NUM
                if pd.Timestamp(d) > LAST_FACT:
                    cell.fill = PLAN_FILL
            r += 1
        if with_wo:
            ws.cell(r, 1, cur), ws.cell(r, 2, prod), ws.cell(r, 3, WO_LABEL)
            vals = wo_piv.loc[(cur, prod)] if (cur, prod) in wo_piv.index else pd.Series(0.0, index=dates_all)
            for c, d in enumerate(dates_all, 4):
                cell = ws.cell(r, c, round(float(vals.iloc[c - 4]) / UNIT, 3))
                cell.number_format, cell.font = NUM, Font(italic=True)
                if pd.Timestamp(d) > LAST_FACT:
                    cell.fill = PLAN_FILL
            for c in range(1, 4):
                ws.cell(r, c).font = Font(italic=True)
            r += 1
    last = r - 1
    # totals: the write-off rows are a flow, not money on balance, so they are summed separately
    totals = [("Итого", "<>" + WO_LABEL), ("Итого списание", WO_LABEL)] if with_wo else [("Итого", None)]
    for label, crit in totals:
        ws.cell(r, 1, label).font = BOLD
        for c in range(4, 4 + len(dates_all)):
            L = get_column_letter(c)
            f = (f'=SUMIF($C${first}:$C${last},"{crit}",{L}{first}:{L}{last})' if crit
                 else f"=SUM({L}{first}:{L}{last})")
            cell = ws.cell(r, c, f)
            cell.number_format, cell.font, cell.fill = NUM, BOLD, TOTAL_FILL
        for c in range(1, 4):
            ws.cell(r, c).fill = TOTAL_FILL
        r += 1
    return r + 2


nxt = block(1, "balance", "ПОРТФЕЛЬ (на балансе; «списание» = остаток списанных, 95413)", with_wo=True)
block(nxt, "reserve", "РЕЗЕРВ (ЦБ)")
for c in range(4, 4 + len(dates_all)):
    ws.column_dimensions[get_column_letter(c)].width = 12
ws.freeze_panes = "D4"


def sheet(name, df, pct=False):
    s = wb.create_sheet(name)
    df = df.reset_index()
    for c, h in enumerate(df.columns, 1):
        cell = s.cell(1, c, " ".join(map(str, h)).strip() if isinstance(h, tuple) else str(h))
        cell.font, cell.fill = Font(bold=True, color="FFFFFF"), HEAD
        s.column_dimensions[get_column_letter(c)].width = 14
    for r, row in enumerate(df.itertuples(index=False), 2):
        for c, v in enumerate(row, 1):
            if isinstance(v, pd.Timestamp):
                v = v.strftime("%d.%m.%Y")
            v = v.item() if hasattr(v, "item") else v
            if isinstance(v, float) and not np.isfinite(v):
                v = None
            cell = s.cell(r, c, v)
            if isinstance(v, float):
                cell.number_format = "0.0" if pct else NUM
    s.freeze_panes = "A2"


mat = M.copy()
mat[FLOW] = mat[FLOW] * 100
mat = mat.rename(columns={**{f"to{j}": f"→ кат. {j}, %" for j in CATEGORIES}, "repaid": "погашено, %",
                          "wo": "списано, %", "source": "откуда строка", "n": "кредитов в окне"})
sheet("Миграция", mat, pct=True)
sheet("Выдачи", iss_view)
sheet("Списанные 95413", wo_view)
sheet("Проблемные кк", cf.set_index("Валюта"))
sheet("Ставки резерва", (R * 100).unstack("category").rename(columns=lambda i: f"кат. {i}, %"), pct=True)
flows = plan_flows.groupby(["cur", "product", "date"])[["new_money", "repaid", "written_off"]].sum().div(UNIT)
flows.insert(0, "issued", [LENDING.get(k, 0.0) / UNIT for k in flows.index])
sheet("Эффект правила списания", eff)
sheet("Потоки плана", flows.rename(columns={"issued": "выдано", "new_money": "новые на балансе", "repaid": "погашено",
                                            "written_off": "списано"}))
if bt is not None:
    sheet("Бэктест", bt)
    sheet("Бэктест по продуктам", bt_prod)
    sheet("Бэктест по месяцам", bt_path, pct=True)
if len(one):
    sheet("Прогноз на 1 месяц", one_err, pct=True)
sheet("Потоки по месяцам", kf.set_index(["cur", "date"]), pct=True)
settings = {k: str(v) for k, v in dict(
    LAST_FACT=f"{LAST_FACT:%d.%m.%Y}", FACT_FROM=FACT_FROM, PLAN_TO=PLAN_TO, MATRIX_MONTHS=MATRIX_MONTHS,
    RATE_MONTHS=RATE_MONTHS, LENDING_MONTHS=LENDING_MONTHS, STOPPED_MONTHS=STOPPED_MONTHS, MIN_LOANS=MIN_LOANS,
    WORSEN=WORSEN, CURE=CURE, RESERVE_RULE=RESERVE_RULE, CBU_MIN_RATE=CBU_MIN_RATE,
    LOAN_LEVEL_DEPTS=LOAN_LEVEL_DEPTS, LOAN_LEVEL_FROM=LOAN_LEVEL_FROM, DAYS_TO_CATEGORY=DAYS_TO_CATEGORY,
    WRITEOFF_DAYS=WRITEOFF_DAYS, WRITEOFF_DAYS_NEW=WRITEOFF_DAYS_NEW, DECISIONS=len(DECISIONS),
    BACKTEST_MONTHS=BACKTEST_MONTHS, BACKTEST_MATRIX_MONTHS=BACKTEST_MATRIX_MONTHS,
    LENDING_FACTOR=LENDING_FACTOR, MANUAL_MOVES=MANUAL_MOVES, WRITEOFF_CHANGE_FROM=WRITEOFF_CHANGE_FROM,
    WRITEOFF_KEEP=WRITEOFF_KEEP, WRITEOFF_RUNOFF_MONTHS=WRITEOFF_RUNOFF_MONTHS, WRITEOFF_RUNOFF=WRITEOFF_RUNOFF,
    RESERVE_RATE_OVERRIDE=RESERVE_RATE_OVERRIDE,
    LENDING_FILE=LENDING_FILE if LENDING_FILE.exists() else "-").items()}
sheet("Настройки", pd.DataFrame({"параметр": list(settings), "значение": list(settings.values())}).set_index("параметр"))
wb.save(OUT_FILE)
print(f"Saved: {OUT_FILE.resolve()}")